# pyvista-manifold examples

Once `pyvista-manifold` is installed, every `pv.PolyData` exposes a `.manifold` accessor: `mesh.manifold.difference(other)`, `mesh.manifold.hull()`, and so on. There is no decorator to apply, no class to import. The accessor registers itself via a PyVista entry point at install time.

This notebook walks through real geometry built with that accessor. The polished renders use PyVista's PBR mode with a CC0 HDR environment from [Poly Haven](https://polyhaven.com/) for image-based lighting; see [the PyVista PBR docs](https://docs.pyvista.org/api/plotting/_autosummary/pyvista.plotter.set_environment_texture) for details.

In [ ]:
import math
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import pyvista as pv

from pyvista_manifold import level_set

# Poly Haven HDR (CC0) used for image-based lighting on every PBR render.
HDRI_SLUG = 'kloofendal_48d_partly_cloudy'
HDRI_RESOLUTION = '1k'
HDRI_CACHE = Path.home() / '.cache' / 'pyvista-manifold' / 'hdri'


def load_hdri() -> pv.Texture:
    """Download (once, cached) and load the IBL HDR as a PyVista texture."""
    HDRI_CACHE.mkdir(parents=True, exist_ok=True)
    target = HDRI_CACHE / f'{HDRI_SLUG}_{HDRI_RESOLUTION}.hdr'
    if not target.exists():
        url = f'https://dl.polyhaven.org/file/ph-assets/HDRIs/hdr/{HDRI_RESOLUTION}/{target.name}'
        tmp = target.with_suffix('.hdr.part')
        try:
            urllib.request.urlretrieve(url, tmp)
            tmp.replace(target)
        except BaseException:
            tmp.unlink(missing_ok=True)
            raise
    tex = pv.read_texture(str(target))
    tex.mipmap = True
    tex.interpolate = True
    return tex


def _z_up_environment_rotation() -> np.ndarray:
    """Return the 3x3 rotation that maps Y-up IBL data to PyVista's Z-up world."""
    return pv.Transform().rotate_x(90).rotation_matrix


def setup_pbr(pl: pv.Plotter) -> None:
    """Configure the active renderer for PBR image-based lighting (IBL only)."""
    pl.remove_all_lights(only_active=True)
    pl.set_environment_texture(
        HDRI, is_srgb=False, rotation=_z_up_environment_rotation(), show_background=False
    )


HDRI = load_hdri()

## Constructive solid geometry

Build a mounting bracket by stacking primitives and subtracting holes. Each step returns a fresh `PolyData`, so the operations chain straight through PyVista's filter pipeline.

In [ ]:
plate = pv.Cube(x_length=8.0, y_length=5.0, z_length=0.6, center=(0, 0, 0.3))
boss = pv.Cylinder(radius=1.6, height=1.6, direction=(0, 0, 1), center=(0, 0, 1.0), resolution=96)
bracket = plate.manifold.union(boss)

for cx, cy in [(-3.2, -1.7), (3.2, -1.7), (-3.2, 1.7), (3.2, 1.7)]:
    hole = pv.Cylinder(
        radius=0.4, height=2.0, direction=(0, 0, 1), center=(cx, cy, 0.5), resolution=48
    )
    bracket = bracket.manifold.difference(hole)

bore = pv.Cylinder(radius=0.7, height=3.0, direction=(0, 0, 1), center=(0, 0, 1.0), resolution=96)
bracket = bracket.manifold.difference(bore)
bracket

In [ ]:
pl = pv.Plotter()
setup_pbr(pl)
pl.add_mesh(
    bracket,
    color='#c2c5c9',
    pbr=True,
    metallic=0.95,
    roughness=0.4,
    smooth_shading=True,
    split_sharp_edges=True,
)
pl.show()

## Triply periodic minimal surfaces

`level_set(f, bounds, edge_length)` extracts an iso-surface from any Python callable. The gyroid below is one of the standard TPMS forms used as 3D-printer infill: continuous, self-supporting, high surface area for given volume.

In [ ]:
def gyroid(x, y, z, scale=2.0):
    s = scale
    return -(
        math.sin(s * x) * math.cos(s * y)
        + math.sin(s * y) * math.cos(s * z)
        + math.sin(s * z) * math.cos(s * x)
    )


tpms = level_set(gyroid, bounds=(-3, -3, -3, 3, 3, 3), edge_length=0.04)
blob = tpms.manifold.intersection(pv.Sphere(radius=2.6, theta_resolution=64, phi_resolution=64))
blob

In [ ]:
pl = pv.Plotter()
setup_pbr(pl)
pl.add_mesh(
    blob,
    color='#d8a23c',
    pbr=True,
    metallic=0.7,
    roughness=0.25,
    smooth_shading=True,
    split_sharp_edges=True,
)
pl.show()

## Lattice infill of a real mesh

Intersect a closed input mesh with a TPMS field. The result is what an FDM slicer might produce for a low-density print: a hollow shell with a continuous gyroid lattice inside.

PyVista's example cow is exported with the y axis as up; rotate it so it stands upright in PyVista's z-up convention.

In [ ]:
cow = pv.examples.download_cow().rotate_x(90)
cow = cow.translate(-np.array(cow.center)).scale(0.6)
lattice_field = level_set(
    lambda x, y, z: gyroid(x, y, z, scale=6.0),
    bounds=(-5, -3, -3, 5, 3, 3),
    edge_length=0.04,
)
infilled = cow.manifold.intersection(lattice_field)
infilled

In [ ]:
pl = pv.Plotter()
pl.enable_anti_aliasing('msaa')
pl.add_mesh(
    infilled,
    color='salmon',
    smooth_shading=True,
    split_sharp_edges=True,
    specular=0.4,
    specular_power=18,
)
pl.add_mesh(cow, color='lightsteelblue', opacity=0.12)
pl.show()

## Topographic slicing

`mesh.manifold.slice_z(z)` returns a closed-polyline `PolyData` for the planar contour at height `z`. Stacking many slices across the bounding box gives the topographic-line look used in CNC toolpath visualization and architectural drawing.

In [ ]:
horse = pv.examples.download_horse()
horse = horse.translate(-np.array(horse.center)).scale(50.0)
z_min, z_max = horse.bounds.z_min, horse.bounds.z_max
heights = np.linspace(z_min + 0.05, z_max - 0.05, 36)
contours = [horse.manifold.slice_z(z) for z in heights]

In [ ]:
pl = pv.Plotter()
for z, c in zip(heights, contours, strict=True):
    c['height'] = np.full(c.n_points, z)
    pl.add_mesh(c, scalars='height', cmap='viridis', line_width=2.5, show_scalar_bar=False)
pl.enable_anti_aliasing('msaa')
pl.show()

## Voronoi-style fracture

`mesh.manifold.split_by_plane(normal, offset)` cuts a solid by an infinite plane and returns both halves. Chain many random cuts of a starting cube and the result is a stack of polyhedral cells, like a 3D Voronoi or a fracture pattern.

In [ ]:
rng = np.random.default_rng(7)
pieces = [pv.Cube(x_length=2.0, y_length=2.0, z_length=2.0)]
for _ in range(7):
    n = rng.normal(size=3)
    n /= np.linalg.norm(n)
    offset = rng.uniform(-0.55, 0.55)
    new_pieces = []
    for p in pieces:
        a, b = p.manifold.split_by_plane(tuple(n), offset)
        if a.n_points:
            new_pieces.append(a)
        if b.n_points:
            new_pieces.append(b)
    pieces = new_pieces
len(pieces)

In [ ]:
pl = pv.Plotter()
for p in pieces:
    centroid = np.array(p.center)
    shifted = p.translate(0.18 * centroid / max(np.linalg.norm(centroid), 1e-6))
    color = rng.uniform(0.45, 0.92, size=3)
    pl.add_mesh(shifted, color=color, smooth_shading=True, split_sharp_edges=True, specular=0.35)
pl.enable_anti_aliasing('msaa')
pl.show()

## Soft edges via Minkowski sum

Minkowski-summing a sharp solid with a small sphere rounds every edge by the sphere's radius. The classic CAD "fillet all edges" operation falls out for free.

In [ ]:
letter = pv.Cube(x_length=4.0, y_length=1.0, z_length=1.0).manifold.union(
    pv.Cube(x_length=1.0, y_length=4.0, z_length=1.0)
)
soft = letter.manifold.minkowski_sum(
    pv.Sphere(radius=0.18, theta_resolution=24, phi_resolution=24)
)
soft

In [ ]:
pl = pv.Plotter(shape=(1, 2))
pl.subplot(0, 0)
pl.add_mesh(letter, color='steelblue')
pl.subplot(0, 1)
setup_pbr(pl)
pl.add_mesh(
    soft,
    color='#7896b8',
    pbr=True,
    metallic=0.6,
    roughness=0.3,
    smooth_shading=True,
    split_sharp_edges=True,
)
pl.link_views()
pl.show()

## Geometry properties for free

The accessor exposes Manifold's exact volume, surface area, and topological genus as plain Python attributes.

In [ ]:
rows = []
for name, m in [
    ('bracket', bracket),
    ('TPMS gyroid', blob),
    ('lattice cow', infilled),
    ('rounded letter', soft),
]:
    rows.append(
        {
            'name': name,
            'volume': m.manifold.volume,
            'surface_area': m.manifold.surface_area,
            'genus': m.manifold.genus,
            'is_valid': m.manifold.is_valid,
            'n_tri': m.manifold.num_tri,
        }
    )
pd.DataFrame(rows)

## Drop down to manifold3d when needed

`mesh.manifold.to_manifold()` returns the raw `manifold3d.Manifold`. Operate on it however you like and convert back with `pyvista_manifold.from_manifold`.

In [ ]:
from pyvista_manifold import from_manifold

raw = pv.Cube().manifold.to_manifold()
shifted = raw.translate((1.5, 0, 0))
from_manifold(raw + shifted).plot(color='steelblue', smooth_shading=True, split_sharp_edges=True)